In [ ]:
import kagglehub

# بناخد الداتا من كاجل ونحملها
path = kagglehub.dataset_download("andreschirinos/p2p-bob-exchange")

print("Path to dataset files:", path)

In [ ]:
import kagglehub
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
import os
import kagglehub
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [ ]:
# بنشوف المجلد الحالي
os.getcwd()   

In [ ]:
# os.listdir()   محتويات الملف

In [ ]:

path = kagglehub.dataset_download("andreschirinos/p2p-bob-exchange")     

In [ ]:

os.listdir(path)   #المحويات الي جوه ملف الي هنشتغل عليه

In [ ]:
print(path)

In [ ]:
df = pd.read_parquet(f"{path}/dashboard_summary.parquet")  #بتختار االفايل من الفولدر الي هتشتغل عليه

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df['timetamp'] = pd.to_datetime(df['timestamp'])  #بنحول من  string to time عشان استفيد منه كداتا ميبقاش مركون

In [ ]:
df.describe()

In [ ]:
df['timetamp'] = pd.to_datetime(df['timetamp'])

In [ ]:
df['source']

In [ ]:
df.sort_index

In [ ]:
df['asset']

In [ ]:
df['asset'].unique()

In [ ]:
df['asset'].value_counts()

#AW

In [ ]:
df.drop(columns= ["source","price_min",
    "price_max",
    "tradablequantity_mean",
    "tradablequantity_min",
    "tradablequantity_max",
    "surplusamount_mean",
    "maxsingletransamount_max",
    "minsingletransamount_min",
    "frequency"
], inplace = True)  #clear the data 

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.isna().sum()  #is any non features in data

In [ ]:
df['price_std'] = df['price_std'].fillna(0)   #std =  standard devision  

In [ ]:
df.isna().sum()  

In [ ]:
df['price_std'].hist(bins = 100, figsize = (8,4))  # pandas use matplotlib hist= 
#use it for draw the data bins = devision the data for num of parts
#fgsize = (width,lenght) for screen

In [ ]:
df['price_std']= np.log1p(df['price_std'])  
# reduce outliars effect and the model don't affect that much from outliarrs  #AW 

In [ ]:
df['price_std'].hist(bins= 100 ,figsize  = (8,4))

In [ ]:


invalid = df["price_std"][~np.isfinite(df['price_std'])]
invalid   #هل فيه ارقام غير صالخه رساضيا  like infinti   | ~ = shit true to fals and opposite
# هو بيطبع ال true

In [ ]:
df.columns


In [ ]:
df['price_mean']

In [ ]:
df['price_mean'].hist(bins = 100, figsize = (8,4))

In [ ]:
df['price_mean'] = np.log1p(df['price_mean'])

In [ ]:
df['price_mean'].hist(bins = 100, figsize = (8,4))

In [ ]:
df.head()

In [ ]:
df = df.drop(index=56451)

In [ ]:
df.describe(percentiles = [0.01,0.05,0.25,0.5,0.75,0.95,0.99])

In [ ]:
df[df['num_transactions'] == 10375]

In [ ]:
df.drop(index = 56422)

In [ ]:
df.describe(percentiles=[0.01,0.05,0.25,0.4,0.5,0.6,0.7,0.85,0.99,])

In [ ]:
df.select_dtypes(include = "number").plot(
    kind = 'box',
    subplots = True,
    layout = (4,3),
    sharex = False,
    figsize= (15,12)

####hello abdulllah Waleed is the best engineer.

)
plt.show()

In [ ]:
outlier_summary = {}  
#dict =  'col':sum num of out liars in one column

for col in df.select_dtypes(include="number").columns: # i make include numbers because outliars is only num
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    #Q1 = 0.25 the all numbers for data
    #Q3= 0.75 the all numbers for data

    IQR = Q3 - Q1  
    #Inter Quartile Range

    lower = Q1 - (1.5 * IQR) #the lowes value not cosiderd as outliar
    upper = Q3 + (1.5 * IQR) #the biggest value not cosiderd as outliar


    outliers = ((df[col] < lower) | (df[col] > upper)).sum()  #  For every column, we count the number of outliers and save the total in the variable outliers

    outlier_summary[col] = outliers # col':outliers appent to outlier_summary  

outlier_summary

In [ ]:



cols_to_log = [
    "price_std",
    "num_transactions",
    "tradablequantity_sum",
    "surplusamount_sum",
    "maxsingletransamount_mean",
    "minsingletransamount_mean",
    "num_ads"
]

for col in cols_to_log:
  df[col] = np.log1p(df[col])

In [ ]:

df.select_dtypes(include="number").plot(
    kind="box",
    subplots=True,
    layout=(4,3),
    figsize=(15,12),
    sharex=False
)

plt.show()

In [ ]:


scaler = StandardScaler()

num_cols = df.select_dtypes(include="number").columns
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
df.head

In [ ]:
df['asset'].value_counts()  


In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df = pd.get_dummies(df,columns = ['asset'])
#get_dummies = itteration for one hot encoding

In [ ]:
df.describe()

In [ ]:
(df["timestamp"] == df["timetamp"]).all()

#بشوف لو العمود متكرر

In [ ]:
df = df.drop(columns=["timetamp"])

In [ ]:
df.columns

In [ ]:
df = df.sort_values('timestamp')
#order by time not index for accurecy related to feature time


In [ ]:
df.describe()

In [ ]:
le_trade = LabelEncoder()

In [ ]:
df["tradetype"] = le_trade.fit_transform(df["tradetype"]) 
# buy,sell to 0,1

In [ ]:
df["tradetype"].head()

In [ ]:
df["tradetype"] = df["tradetype"].astype("float32")

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df["day"] = df["timestamp"].dt.day
df["weekday"] = df["timestamp"].dt.weekday
df["month"] = df["timestamp"].dt.month
df["hour"] = df["timestamp"].dt.hour 
df["minute"] = df["timestamp"].dt.minute
#insted of use time the model can't read make column for each time to be readable and model can use it will
#AW

In [ ]:
df.drop(columns=["timestamp"], inplace=True)

In [ ]:
model = XGBClassifier(
    objective="binary:logistic", #the output type binary = 0 | 1
    eval_metric="logloss", #teacher see how student result by percent
    n_estimators=200,#how nuch trees
    max_depth=6,# for each tree num of quiesitons
    learning_rate=0.05,#each tree how much learn from brevious tree
    random_state=42 # any number works, it just makes the results reproducible so accuracy doesn’t change on each run.
)

In [ ]:
df.describe()

In [ ]:
df["target"] = (df["price_mean"].shift(-1) > df["price_mean"]).astype(int) #bool to int
df = df.dropna()
#shift to up and clean last empty cell 
#prediction use time

In [ ]:
split = int(len(df)*0.8)
train = df.iloc[:split]
test = df.iloc[split:]

# integer location = iloc split by index
# show len of data then split it by index

In [ ]:
x_train = train.drop(columns = ['target'])
y_train = train['target']

x_test = test.drop(columns= ['target'])
y_test = test['target']

In [ ]:
print(y_train.unique())

In [ ]:
model.fit(x_train, y_train)

In [ ]:
prediction_For_Model = model.predict(x_test)

In [ ]:


accuracy_score(y_test, prediction_For_Model)

In [ ]:
y_test.value_counts(normalize=True) 
#show the data is bias and return percentage

In [ ]:

print(classification_report(y_test, prediction_For_Model))
#show accuarecy more than option